# Variables Categoricas

Las variables categoricas son aquellas que tambien se consideran cualitativas, como adejetivos. Supongamos que queremos ver el color que la gente prefiere vestir cuando va a una fiesta: 
* Azul
* Verde
* Rojo
Estos colores no se representan como tal en un DataFrame, sino, que cada color sera un atributo que tendra un 1 o un 0, verdadero o falso, en caso de cumplir con tal color.

In [1]:
color = {"azul": 1, "verde": 0, "rojo": 0}

De esta forma es como una computadora toma en cuenta el tipo de atributo que se esta analizando. Como no estamos tratando con valores continuos, es decir numeros, entonces la prediccion tambien vendra en valores booleanos, que de hecho es un poco mas critico que los continuos.

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv("data/melb_data.csv")

y = data.Price
X = data.drop(['Price'], axis = 1)

# eliminamos las columnas con valores perdidos
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state = 0)
cols_with_missing = [col for col in X_train_full.columns if X_train_full[col].isnull().any()]
X_train_full.drop(cols_with_missing, axis = 1, inplace = True)
X_valid_full.drop(cols_with_missing, axis = 1, inplace = True)

# Cardinalidad: es el numero de valores unicos en una columna
# baja cardinalidad: pocos valores unicos
# alta cardinalidad: muchos valores unicos
# seleccionamos los valores con pocos valores unicos
low_cardinality_cols = [cname for cname in X_train_full.columns if X_train_full[cname].nunique() < 10 and X_train_full[cname].dtype == "object"]

# seleccionamos las columnas numericas
numerical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].dtype in ['int64', 'float']]

# conservamos las columnas seleccionadas
my_cols = low_cardinality_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()

In [13]:
s = (X_train.dtypes == 'object')
object_cols = list(s[s].index)

print("variables categoricas: ", object_cols)

variables categoricas:  ['Type', 'Method', 'Regionname']


In [5]:
# definimos las funciones para comparar aproximaciones
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# funcoin para comparar diferentes aproximaciones
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators = 100, random_state = 0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

Por ahora, manejaremos con variables categoricas para alimentar un modelo para obtener una prediccion de variable continua.

## Eliminamos las variables categoricas

Aunque suene raro, eliminar las variables categoricas tambien es una forma de tratar con ellas, pero como siempre, esto depende del caso.

In [8]:
drop_X_train = X_train.select_dtypes(exclude = ['object'])
drop_X_valid = X_valid.select_dtypes(exclude = ['object'])

print("MAE de aproximacion 1: ", score_dataset(drop_X_train, drop_X_valid, y_train, y_valid)) 

MAE de aproximacion 1:  175703.48185157913


## Codificacion ordinaria

Este tipo de codificacion es simple, sustituye los valores categoricos por valores enteros, como azul: 1, verde: 2, rojo: 3. de esta forma se hace un analizis un poco mas profundo pero simple.

In [15]:
from sklearn.preprocessing import OrdinalEncoder

# hacemos una compia para evitar cambios en el df original
label_X_train = X_train.copy()
label_X_valid = X_valid.copy()

# aplicamos codificador ordinario
ordinal_encoder = OrdinalEncoder()
label_X_train[object_cols] = ordinal_encoder.fit_transform(X_train[object_cols])
label_X_valid[object_cols] = ordinal_encoder.transform(X_valid[object_cols])

print("MAE de la aproximacion 2: ", score_dataset(label_X_train, label_X_valid, y_train, y_valid))

MAE de la aproximacion 2:  165936.40548390493


## One-Hot Encoding (codificacion en caliente)

Este otro tipo de codificacion lo que hace es darle una columna a cada valor de una columna categorica que hay. Ejemplo: supongamos que tenemos una columna llamada color, y tiene 5 valore unicos, entonces cadaa valor unico pasara a ser una columna nueva que almacenara valores tipo bool cada que se cumpla con tal valor.

In [18]:
from sklearn.preprocessing import OneHotEncoder

# aplicamos OH a cada columna categorica que ya tenemos
OH_encoder = OneHotEncoder(handle_unknown = "ignore", sparse_output = False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[object_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[object_cols]))

# El OHE remueve los indices, asi que los colocamos de nuevo
OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

# removemos las columnas categoricas que teniamos para remplazarlas
num_X_train = X_train.drop(object_cols, axis = 1)
num_X_valid = X_valid.drop(object_cols, axis = 1)

# añadimos OHE
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis = 1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis = 1)

# Nos aseguramos que las columnas son de tipo string
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

print("MAE de la aproximacion 3: ", score_dataset(OH_X_train, OH_X_valid, y_train, y_valid))

MAE de la aproximacion 3:  166089.4893009678


Y aunque el analizis es mas profunco que la codificacion ordinaria, la aproximacion es apenas mas grande que el resultado de OE.